In [2]:
# ============================================================================
# 03b_inter_judge.ipynb  —  Inter-judge agreement (secondary model re-scoring)
# ----------------------------------------------------------------------------
# Single-cell notebook. All code and comments are in English.
# Run AFTER 03_judge_metric (needs data/scored.parquet). Writes
# results/tables/tab_interjudge.csv for the paper's reliability row.
#
# PURPOSE
#   The primary judge (STRONG tier) scored every output in 03. Here a SECONDARY
#   judge from a different, smaller tier re-scores a random sample of the same
#   outputs under the IDENTICAL rubric, and we quantify agreement on the
#   qualitative-adequacy axis with:
#     - Spearman's rho                    (rank correlation)
#     - quadratic-weighted Cohen's kappa  (credits near-misses on the 1-5 scale)
#
# INPUT   data/scored.parquet   (primary judge scores from 03)
# OUTPUT  data/interjudge.parquet, results/tables/tab_interjudge.csv
# ============================================================================
import sys, os, json, warnings
from pathlib import Path
import numpy as np
import pandas as pd

_HERE = Path.cwd()
_ROOT = _HERE if (_HERE / "src").exists() else _HERE.parent
sys.path.insert(0, str(_ROOT / "src"))
from common import (DATA_DIR, TABLES_DIR, load_env, require_input, RANDOM_SEED)

warnings.filterwarnings("ignore")
np.random.seed(RANDOM_SEED)

from scipy.stats import spearmanr
from sklearn.metrics import cohen_kappa_score

cfg = load_env()
from openai import OpenAI
client = OpenAI(api_key=cfg["api_key"])
JUDGE_SECONDARY = cfg["mid"]          # different (smaller) tier than the primary (strong)

# ---- sample size for the reliability check (raise if budget allows) ----------
SAMPLE_N = 60

# Re-score the primary judge's INFORMED scores on the GAP-GIVEN outputs, i.e. the
# same cells used for the headline analysis, so the agreement speaks to the
# scores the paper actually reports.
s = pd.read_parquet(require_input(DATA_DIR / "scored.parquet",
                                  hint="Run 03_judge_metric first."))
base = s[(s.judge_cond == "informed") & (s.condition == "gap_given")].copy()
base = base.dropna(subset=["judge_qual"]).reset_index(drop=True)

# To re-score we need the text that was judged. If 03 did not persist
# rationale/rating/delta into scored.parquet, join them from perturbed.parquet.
need = {"rationale", "rating", "delta"}
if not need.issubset(base.columns):
    pert = pd.read_parquet(require_input(DATA_DIR / "perturbed.parquet",
                                         hint="Run 02_perturbation first."))
    keep = ["case_id", "condition", "pert", "k", "rating", "rationale", "delta"]
    base = base.merge(pert[keep], on=["case_id", "condition", "pert", "k"],
                      how="left", suffixes=("", "_p"))
    for c in ("rationale", "rating", "delta"):
        if c not in base.columns and f"{c}_p" in base.columns:
            base[c] = base[f"{c}_p"]

# Random sample of outputs to re-score.
samp = base.sample(n=min(SAMPLE_N, len(base)), random_state=RANDOM_SEED).copy()

def judge_secondary(rationale, rating, delta, model):
    # Same rubric and framing as the primary INFORMED judge in 03.
    gap_ctx = (f"Context: the agency rating is {int(delta):+d} notches vs the "
               f"quantitative profile; a good rationale presents a qualitative "
               f"justification whose direction matches this gap.")
    sys_msg = ("You are evaluating a credit rating rationale. Score five axes "
               "1-5: grade_accuracy, grade_rationale_consistency, "
               "qualitative_adequacy, factual_faithfulness, explainability. "
               "qualitative_adequacy: 5=clear & consistent with the gap's "
               "direction, 3=present but generic, 1=absent or contradicting.")
    user = (f"{gap_ctx}\nRating: {rating}\nRationale:\n{rationale}\n\n"
            f"Return JSON with the five integer axes.")
    r = client.chat.completions.create(model=model, temperature=0.0,
        messages=[{"role": "system", "content": sys_msg},
                  {"role": "user", "content": user}],
        response_format={"type": "json_object"})
    return json.loads(r.choices[0].message.content)

sec_qual = []
for _, row in samp.iterrows():
    try:
        ax = judge_secondary(row.get("rationale", ""), row.get("rating", ""),
                             int(row.get("delta", 0)), JUDGE_SECONDARY)
        sec_qual.append(ax.get("qualitative_adequacy", np.nan))
    except Exception:
        sec_qual.append(np.nan)
samp["judge_qual_secondary"] = sec_qual

# Pair primary vs secondary on the qualitative-adequacy axis.
pair = samp.dropna(subset=["judge_qual", "judge_qual_secondary"]).copy()
pair["p"] = pair["judge_qual"].round().astype(int).clip(1, 5)
pair["q"] = pair["judge_qual_secondary"].round().astype(int).clip(1, 5)

rho, rho_p = spearmanr(pair["judge_qual"], pair["judge_qual_secondary"])
kappa_qw = cohen_kappa_score(pair["p"], pair["q"], weights="quadratic",
                             labels=[1, 2, 3, 4, 5])

out = pd.DataFrame({
    "metric": ["n_pairs", "spearman_rho", "spearman_p", "quadratic_weighted_kappa"],
    "value":  [len(pair), round(float(rho), 3), round(float(rho_p), 4),
               round(float(kappa_qw), 3)]
})
samp.to_parquet(DATA_DIR / "interjudge.parquet", index=False)
out.to_csv(TABLES_DIR / "tab_interjudge.csv", index=False, encoding="utf-8")
print(out.to_string(index=False))
print(f"\nSecondary judge tier: {JUDGE_SECONDARY} | sample: {len(pair)} pairs")

                  metric  value
                 n_pairs 60.000
            spearman_rho  0.555
              spearman_p  0.000
quadratic_weighted_kappa  0.553

Secondary judge tier: gpt-5.4-mini | sample: 60 pairs
